<a href="https://colab.research.google.com/github/siriusGK/Codegk/blob/main/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Using GCN and GAT for the 

In [ ]:
!pip install torch-geometric



In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
data = np.load("/content/drive/MyDrive/QG_jets_1.npz")

# Example keys (depends on dataset)
print(data.files)

X = data["X"]       # jet particles
y = data["y"]       # labels (0=gluon, 1=quark)

print("Data shape:", X.shape)
print("Labels shape:", y.shape)

['X', 'y']
Data shape: (100000, 134, 4)
Labels shape: (100000,)


In [ ]:
def knn_graph(nodes, k=8):

    num_nodes = nodes.shape[0]

    eta = nodes[:,1]#taking the eta value from nodes feature that is2nd column
    phi = nodes[:,2]#taking the phi value from nodes feature that is 3rd column

    edges = []

    for i in range(num_nodes):

        dist = (eta - eta[i])**2 + (phi - phi[i])**2#calculating the distance between the nodes using eta and phi values

        neighbors = np.argsort(dist)[1:k+1]#getting the indices of the k nearest neighbors (excluding itself)

        for j in neighbors:
            edges.append([i,j])#adding the edge between the node and its neighbors

    edge_index = torch.tensor(edges).t().contiguous()#converting the edge to tensor

    return edge_index

In [ ]:
graphs = []

for jet, label in zip(X, y):#zip the x and y labeltogether to iterate 

    jet = jet[jet[:,0] > 0]   # remove zero padding

    node_features = torch.tensor(jet, dtype=torch.float)#converting the jet data to tensor

    edge_index = knn_graph(jet)

    graph = Data(
        x=node_features,
        edge_index=edge_index,
        y=torch.tensor([label], dtype=torch.long)
    )

    graphs.append(graph)

In [ ]:
train_graphs, test_graphs = train_test_split(
    graphs,
    test_size=0.2,
    random_state=42#to initiale so that same value appers in case if we run same code
)

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)#loading data in batch size of 32 with suffleing the data to make random
test_loader  = DataLoader(test_graphs, batch_size=32)#loading data in batch size of 32 without suffleing the data to make random

In [ ]:
class GCNNet(nn.Module):

    def __init__(self, in_channels):

        super().__init__()

        self.conv1 = GCNConv(in_channels, 64)#convolutinf the input data to 64 channels
        self.conv2 = GCNConv(64, 64)#convolution of the 64 channels to 64 channels

        self.fc = nn.Linear(64,2)#making liner layer which give 2 output for 2 classes that is gluon and quark

    def forward(self, data):#forward pass to move through the model

        x, edge_index, batch = data.x, data.edge_index, data.batch#

        x = self.conv1(x, edge_index)#apply 1st convolution on the input
        x = F.relu(x)#activation function to add non-linearity(means <0 makes 0 and >0 keep as it is)

        x = self.conv2(x, edge_index)#why after 1st. since we have 64 channels we can apply 2nd convolution to get more complex features
        x = F.relu(x)

        x = global_mean_pool(x, batch)#averthe the feature nodes to resude the feature dimention

        x = self.fc(x)#calculate the output of the model

        return x

In [ ]:
class GATNet(nn.Module):

    def __init__(self, in_channels):#definig the gat net class

        super().__init__()#super class constructor (that is nn.module) calling

        self.conv1 = GATConv(in_channels, 64, heads=4)

        self.conv2 = GATConv(64*4, 64)

        self.fc = nn.Linear(64,2)

    def forward(self, data):

        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = F.elu(x)

        x = self.conv2(x, edge_index)
        x = F.elu(x)

        x = global_mean_pool(x, batch)

        x = self.fc(x)

        return x

In [ ]:
def train(model, loader, optimizer):

    model.train()

    total_loss = 0

    for batch in loader:

        optimizer.zero_grad()

        out = model(batch)

        loss = F.cross_entropy(out, batch.y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:

def evaluate(model, loader):

    model.eval()

    y_true = []#making arreys to fill the true lable
    y_pred = []#making arreys to fill the predicted lable
    y_prob = []#making arreys to fill the predicted probability

    with torch.no_grad():

        for batch in loader:#uses to iterrate through the data in batches

            out = model(batch)#to get output fron the baches model

            prob = F.softmax(out, dim=1)

            pred = prob.argmax(dim=1)

            y_true.extend(batch.y.cpu().numpy())#to convert tensor data to numpy by moving to cpu
            y_pred.extend(pred.cpu().numpy())
            y_prob.extend(prob[:,1].cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    return acc, auc


In [ ]:
in_features = graphs[0].num_node_features

gcn = GCNNet(in_features)
gat = GATNet(in_features)

optimizer_gcn = torch.optim.Adam(gcn.parameters(), lr=0.01)
optimizer_gat = torch.optim.Adam(gat.parameters(), lr=0.01)

epochs = 20

print("Training GCN")

for epoch in range(epochs):

    loss = train(gcn, train_loader, optimizer_gcn)

    acc, auc = evaluate(gcn, test_loader)

    print(f"Epoch {epoch} | Loss {loss:.4f} | Acc {acc:.4f} | AUC {auc:.4f}")


print("Training GAT")

for epoch in range(epochs):

    loss = train(gat, train_loader, optimizer_gat)

    acc, auc = evaluate(gat, test_loader)

    print(f"Epoch {epoch} | Loss {loss:.4f} | Acc {acc:.4f} | AUC {auc:.4f}")

Training GCN
Epoch 0 | Loss 0.5695 | Acc 0.7606 | AUC 0.8279
Epoch 1 | Loss 0.5183 | Acc 0.7769 | AUC 0.8458
Epoch 2 | Loss 0.5127 | Acc 0.7721 | AUC 0.8431
Epoch 3 | Loss 0.5122 | Acc 0.7673 | AUC 0.8379
Epoch 4 | Loss 0.5114 | Acc 0.7693 | AUC 0.8428
Epoch 5 | Loss 0.5054 | Acc 0.7358 | AUC 0.8453
Epoch 6 | Loss 0.5028 | Acc 0.7826 | AUC 0.8480
Epoch 7 | Loss 0.5019 | Acc 0.7854 | AUC 0.8514
Epoch 8 | Loss 0.4996 | Acc 0.7857 | AUC 0.8531
Epoch 9 | Loss 0.4996 | Acc 0.7809 | AUC 0.8506
Epoch 10 | Loss 0.4994 | Acc 0.7863 | AUC 0.8505
Epoch 11 | Loss 0.4970 | Acc 0.7819 | AUC 0.8520
Epoch 12 | Loss 0.4978 | Acc 0.7798 | AUC 0.8519
Epoch 13 | Loss 0.4982 | Acc 0.7870 | AUC 0.8521
Epoch 14 | Loss 0.4964 | Acc 0.7611 | AUC 0.8510
Epoch 15 | Loss 0.4973 | Acc 0.7862 | AUC 0.8527
Epoch 16 | Loss 0.4957 | Acc 0.7875 | AUC 0.8540
Epoch 17 | Loss 0.4968 | Acc 0.7831 | AUC 0.8494
Epoch 18 | Loss 0.4980 | Acc 0.7594 | AUC 0.8313
Epoch 19 | Loss 0.4971 | Acc 0.7803 | AUC 0.8522
Training GAT
Epoc

Considerations I have taken to project this point-cloud dataset to a set of interconnected nodes and edges are 
i) Each particle in the jet is treated as a node with features (pT, eta, phi).
ii) Edges are created using a k-nearest neighbor approach in the (eta, phi) space to capture local interactions.
iii) This allows the GNN to learn from both individual particle properties and their relationships, improving classification performance.

Graph Convolution Network(GCN)
#Result-Final result after 20th epochs Loss 0.4971 | Acc 0.7803 | AUC 0.8522

Graph Attention Network (GAT)
Final result after 20th epoch Loss 0.7135 | Acc 0.5004 | AUC 0.5000
